In [1]:
import numpy as np
import nidaqmx

In [2]:
def list_devices():
    """ List the devices currently connected to the system. """

    return [ d.name for d in nidaqmx.system.System().devices ]


def list_analog_inputs():
    """ List the analog inputs for each device """

    channels = dict()
    for dev in nidaqmx.system.System().devices:
        channels[dev.name] = dev.ai_physical_chans

    return channels


def list_analog_outputs():
    """ List the analog outputs for each device """

    channels = dict()
    for dev in nidaqmx.system.System().devices:
        channels[dev.name] = dev.ao_physical_chans

    return channels


def list_boolean_inputs():
    """ List the boolean inputs for each device """

    channels = dict()
    for dev in nidaqmx.system.System().devices:
        channels[dev.name] = dev.di_lines

    return channels


def list_boolean_outputs():
    """ List the boolean outputs for each device """

    channels = dict()
    for dev in nidaqmx.system.System().devices:
        channels[dev.name] = dev.do_lines

    return channels

In [3]:
niDevice = list_devices()
nidaq_device = nidaqmx.system.Device(niDevice[0])
clock_channel="/Dev1/PFI0"

In [50]:
fsamp = 25000
# values = [0, 1, 0, 2, 0, 4, 0, 8, 0, 34, 0, 64, 0, 128, 0, 255]
values = [0, 255, 0, 255, 0, 255, 0, 255, 0, 255, 0, 255, 0, 255]

for val in values:
    dig_data = np.array([4]*fsamp, dtype=np.uint32)  # One second of data
    
    # make 1 s of data
    val_hex = val.to_bytes(1, 'little')
    val_hexuint32 = val_hex+val_hex+val_hex+val_hex
    val_uint32 = int.from_bytes(val_hexuint32, byteorder='little', signed=False)
    dig_data = np.array([val_uint32]*int(fsamp/4), dtype=np.uint32)

    channel_bool = 'Dev1/port0'
    dig_task = nidaqmx.Task()
    dig_task.do_channels.add_do_chan(channel_bool, line_grouping=nidaqmx.constants.LineGrouping.CHAN_FOR_ALL_LINES)
    dig_task.timing.cfg_samp_clk_timing( rate=fsamp,
                                sample_mode=nidaqmx.constants.AcquisitionType.FINITE,
                                samps_per_chan=fsamp)
    


        
    dig_task.write(dig_data, auto_start=False)
    dig_task.start()
    dig_task.wait_until_done(timeout=20.0)
    dig_task.close()
